# Consolidated null-results and provenance tables

**Objective.** Surface the research discipline already embedded in the
repository. The results table lists every machine-readable pre-declared BH/FDR
family through Notebook 74. The provenance table maps each current core
write-up number, plus every family source artifact, to its notebook, hashes,
frozen specification and seed/status boundary.

**Status.** Aggregate-only extraction. Notebook 75 remains unexecuted and is
excluded. This notebook computes no new return, coefficient, p-value or gate.

In [1]:
from __future__ import annotations

import hashlib
import json
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "final_experiments":
    ROOT = ROOT.parent

OUTPUT = ROOT / "final_experiments/outputs/78_null_results_and_provenance_tables"
SPEC_PATH = ROOT / "final_experiments/frozen_specs/examiner_facing_analysis_pack_v1_20260812.json"
MULTIPLICITY_TEX = ROOT / "dissertation/tables/tab_multiplicity_families.tex"
PROVENANCE_TEX = ROOT / "dissertation/tables/tab_core_result_provenance.tex"
OUTPUT.mkdir(parents=True, exist_ok=True)
MULTIPLICITY_TEX.parent.mkdir(parents=True, exist_ok=True)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def notebook_number(path: Path) -> int | None:
    match = re.match(r"(\d{2})_", path.name)
    return int(match.group(1)) if match else None


def notebook_for_artifact(path: Path) -> tuple[str, int | None]:
    name = path.parent.name
    number = notebook_number(Path(name))
    notebooks = sorted((ROOT / "final_experiments").glob(f"{name}.ipynb"))
    return (str(notebooks[0].relative_to(ROOT)) if notebooks else "", number)


def seeds_from_manifest(manifest: dict[str, object]) -> str:
    values: list[str] = []

    def walk(value: object, key: str = "") -> None:
        if isinstance(value, dict):
            for child_key, child in value.items():
                walk(child, child_key)
        elif "seed" in key.lower() and isinstance(value, (int, float, str)):
            token = str(value)
            if token not in values:
                values.append(token)

    walk(manifest)
    return ";".join(values) if values else "none recorded / deterministic"


def tex_escape(value: object) -> str:
    text = str(value)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in text)


def markdown_table(frame: pd.DataFrame) -> str:
    """Small dependency-free pipe table for aggregate appendix previews."""
    safe = frame.fillna("").astype(str).apply(lambda column: column.str.replace("|", r"\|", regex=False))
    header = "| " + " | ".join(safe.columns) + " |"
    divider = "| " + " | ".join("---" for _ in safe.columns) + " |"
    rows = ["| " + " | ".join(row) + " |" for row in safe.itertuples(index=False, name=None)]
    return "\n".join([header, divider, *rows])


spec = json.loads(SPEC_PATH.read_text())
assert "75" in spec["notebook_78"]["multiplicity_scope"]

## Discover every machine-readable BH/FDR family

Labeled artifacts are discovered automatically. Earlier/later notebooks that
saved BH decisions without repeating the family label in every row are added
through the explicit registry below. That registry specifies row filters, so
temporal sensitivities and gate summaries cannot be mistaken for test-family
members. Repeated presentations of the same exact family label are
deduplicated by retaining the earliest numbered source.

In [2]:
family_rows: list[dict[str, object]] = []


def normalise_decisions(values: pd.Series, *, kind: str) -> pd.Series:
    values = values.dropna()
    if kind == "q_value":
        return pd.to_numeric(values, errors="raise").le(0.05)
    return values.map(
        lambda value: value
        if isinstance(value, (bool, np.bool_))
        else str(value).strip().lower() in {"true", "1", "yes"}
    ).astype(bool)


def add_family(
    *,
    path: Path,
    frame: pd.DataFrame,
    family: str,
    decision_column: str,
    decision_kind: str = "boolean",
) -> None:
    decisions = normalise_decisions(frame[decision_column], kind=decision_kind)
    if decisions.empty:
        return
    notebook_path, number = notebook_for_artifact(path)
    if number is None or number > 74:
        return
    manifest_path = path.parent / "manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    family_rows.append(
        {
            "notebook_number": number,
            "notebook": notebook_path,
            "source_artifact": str(path.relative_to(ROOT)),
            "family": family,
            "decision_column": decision_column,
            "tests": int(len(decisions)),
            "bh_survivors": int(decisions.sum()),
            "all_null": bool(decisions.sum() == 0),
            "source_sha256": sha256(path),
            "manifest": str(manifest_path.relative_to(ROOT)) if manifest_path.exists() else "",
            "manifest_sha256": sha256(manifest_path) if manifest_path.exists() else "",
            "seed": seeds_from_manifest(manifest) if manifest else "none recorded / deterministic",
            "status": str(manifest.get("status", manifest.get("analysis_status", "aggregate result"))),
        }
    )


for path in sorted((ROOT / "final_experiments/outputs").glob("*/*.csv")):
    number = notebook_number(path.parent)
    if number is None or number > 74:
        continue
    try:
        frame = pd.read_csv(path)
    except Exception:
        continue
    family_columns = [column for column in ("multiplicity_family", "bh_family") if column in frame.columns]
    reject_columns = [
        column
        for column in frame.columns
        if "bh_reject" in column.lower() or "reject_q05" in column.lower()
    ]
    q_columns = [
        column
        for column in frame.columns
        if column.lower() in {"q_bh", "bh_q_value", "q_value", "q_bh_two_test"}
        or column.lower().endswith("_q_bh")
    ]
    decision_columns = [(column, "boolean") for column in reject_columns]
    if not decision_columns:
        decision_columns = [(column, "q_value") for column in q_columns]
    if not family_columns or not decision_columns:
        continue
    for family_column in family_columns:
        for family, group in frame.dropna(subset=[family_column]).groupby(family_column, sort=False):
            family_text = str(family)
            if family_text.lower().startswith("descriptive_"):
                continue
            for decision_column, decision_kind in decision_columns:
                add_family(
                    path=path,
                    frame=group,
                    family=family_text,
                    decision_column=decision_column,
                    decision_kind=decision_kind,
                )


def subset(frame: pd.DataFrame, filters: dict[str, object]) -> pd.DataFrame:
    selected = frame
    for column, value in filters.items():
        selected = selected.loc[selected[column].eq(value)]
    return selected


explicit_families = [
    ("19_fnspid_sentiment_risk_overlay/paired_inference.csv", "Development sentiment-risk overlays", "bh_reject_q05", "boolean", {"split": "development"}),
    ("19_fnspid_sentiment_risk_overlay/paired_inference.csv", "Evaluation sentiment-risk overlays", "bh_reject_q05", "boolean", {"split": "evaluation"}),
    ("20_fnspid_market_sentiment_risk_overlay/paired_inference.csv", "Development market-sentiment net-return overlays", "bh_reject_q05", "boolean", {"split": "development", "outcome": "net_return"}),
    ("20_fnspid_market_sentiment_risk_overlay/paired_inference.csv", "Development market-sentiment downside overlays", "bh_reject_q05", "boolean", {"split": "development", "outcome": "downside_sq"}),
    ("20_fnspid_market_sentiment_risk_overlay/paired_inference.csv", "Evaluation market-sentiment net-return overlays", "bh_reject_q05", "boolean", {"split": "evaluation", "outcome": "net_return"}),
    ("20_fnspid_market_sentiment_risk_overlay/paired_inference.csv", "Evaluation market-sentiment downside overlays", "bh_reject_q05", "boolean", {"split": "evaluation", "outcome": "downside_sq"}),
    ("26_lseg_gemma_sparse_risk_brakes/paired_inference.csv", "LSEG sparse-brake full-window return and downside", "bh_reject_q05", "boolean", {"period": "full"}),
    ("27_fnspid_har_sentiment_composition/paired_inference.csv", "HAR composition net-return comparisons", "bh_reject_q05", "boolean", {"outcome": "net_return"}),
    ("27_fnspid_har_sentiment_composition/paired_inference.csv", "HAR composition downside comparisons", "bh_reject_q05", "boolean", {"outcome": "downside_sq"}),
    ("28_lseg_gemma_sparse_extreme_rotation/paired_inference.csv", "LSEG extreme-rotation return comparisons", "bh_reject_q05", "boolean", {"outcome": "net_return"}),
    ("28_lseg_gemma_sparse_extreme_rotation/paired_inference.csv", "LSEG extreme-rotation downside comparisons", "bh_reject_q05", "boolean", {"outcome": "downside_sq"}),
    ("30_lseg_gemma_capped_extrema_spread/paired_inference.csv", "Capped-extrema return and downside", "bh_reject_q05", "boolean", {}),
    ("31_lseg_gemma_sector_neutral_extrema_spread/paired_inference.csv", "Sector-neutral extrema return and downside", "bh_reject_q05", "boolean", {}),
    ("32_lseg_gemma_between_sector_extrema/paired_inference.csv", "Between-sector extrema return and downside", "bh_reject_q05", "boolean", {}),
    ("33_lseg_gemma_original33_universe_robustness/paired_inference.csv", "Original-33 between-sector return and downside", "bh_reject_q05", "boolean", {}),
    ("33_lseg_gemma_original33_universe_robustness/rule_vs_cash.csv", "Original-33 rules versus cash", "bh_reject_q05", "boolean", {}),
    ("34_lseg_gemma_finbert_matched_count_substitution/paired_inference.csv", "Matched-count scorer substitutions", "bh_reject_q05", "boolean", {}),
    ("35_lseg_gemma_event_timing_circular_shift/timing_inference.csv", "Event-timing circular-shift net returns", "bh_reject_net_q05", "boolean", {}),
    ("40_fnspid_lseg_gemma_continuous_rule_transfer/mechanism_inference.csv", "Continuous-rule transfer mechanisms", "q_bh", "q_value", {}),
    ("42_fnspid_2010_preperiod_transfer/mechanism_inference.csv", "2010 pre-period transfer mechanisms", "q_bh", "q_value", {}),
    ("45_sentiment_downside_timing_placebo/timing_results.csv", "Downside timing placebos across regimes", "downside_q_bh", "q_value", {}),
    ("45_sentiment_downside_timing_placebo/timing_results.csv", "Return timing placebos across regimes", "return_q_bh", "q_value", {}),
    ("46_lseg_external_time_har_base_transfer/paired_inference.csv", "Gemma overlay return and downside", "q_bh_two_test", "q_value", {"comparison": "gemma_overlay_vs_frozen_har"}),
    ("50_lseg_gemma_hybrid_inverse_volatility/paired_inference.csv", "Inverse-volatility hybrid comparisons", "bh_reject_q05", "boolean", {}),
    ("59_lseg_gemma_hybrid_session_decomposition/component_inference.csv", "Hybrid session component means", "bh_reject", "boolean", {}),
    ("60_lseg_gemma_hybrid_long_leg_exposure_control/control_inference.csv", "Long-leg exposure-control differences", "bh_reject", "boolean", {}),
    ("61_lseg_gemma_hybrid_long_leg_sector_component/component_inference.csv", "Long-leg sector component differences", "bh_reject", "boolean", {}),
    ("64_lseg_gemma_hybrid_long_leg_residual_portfolios/primary_inference.csv", "Residual-portfolio primary comparisons", "bh_reject", "boolean", {}),
    ("64_lseg_gemma_hybrid_long_leg_residual_portfolios/factor_models.csv", "Residual-portfolio factor intercepts", "bh_reject", "boolean", {}),
    ("65_lseg_gemma_hybrid_specificity_filter/primary_inference.csv", "Specificity-filter primary comparisons", "bh_reject", "boolean", {}),
    ("67_lseg_gemma_finbert_joined_long_window/mechanism_inference.csv", "Joined-window mechanism tests", "bh_reject_q05", "boolean", {}),
    ("69_lseg_target_materiality_incremental_value/primary_family.csv", "Target-materiality primary directional family", "reject_q05", "boolean", {}),
    ("69_lseg_target_materiality_incremental_value/reaction_magnitude_regressions.csv", "Target-materiality reaction regressions", "reject_q05", "boolean", {}),
    ("69_lseg_target_materiality_incremental_value/reaction_magnitude_oos.csv", "Target-materiality reaction out-of-sample tests", "reject_q05", "boolean", {}),
    ("69_lseg_target_materiality_incremental_value/horizon_matched.csv", "Target-materiality horizon-matched tests", "reject_q05", "boolean", {}),
    ("72_lseg_negative_pressure_har_transfer/primary_inference.csv", "Recent negative-pressure HAR return and downside", "bh_reject_q05", "boolean", {}),
    ("72_lseg_negative_pressure_har_transfer/timing_tests.csv", "Recent negative-pressure HAR timing tests", "q_bh_two_test", "q_value", {}),
    ("73_lseg_negative_pressure_har_backward_transfer/primary_inference.csv", "Backward negative-pressure HAR return and downside", "bh_reject_q05", "boolean", {}),
    ("73_lseg_negative_pressure_har_backward_transfer/timing_tests.csv", "Backward negative-pressure HAR timing tests", "q_bh_two_test", "q_value", {}),
]

for relative, family, decision_column, decision_kind, filters in explicit_families:
    path = ROOT / "final_experiments/outputs" / relative
    if not path.exists():
        raise FileNotFoundError(path)
    frame = subset(pd.read_csv(path), filters)
    add_family(
        path=path,
        frame=frame,
        family=family,
        decision_column=decision_column,
        decision_kind=decision_kind,
    )

multiplicity = (
    pd.DataFrame(family_rows)
    .sort_values(["notebook_number", "source_artifact", "family"])
    .drop_duplicates(["family"], keep="first")
    .reset_index(drop=True)
)
if multiplicity.empty:
    raise RuntimeError("no machine-readable multiplicity families were discovered")
if (multiplicity["notebook_number"] == 75).any():
    raise RuntimeError("the sealed Notebook 75 entered the multiplicity ledger")
display(multiplicity)

family_summary = pd.DataFrame(
    [
        {
            "families": len(multiplicity),
            "tests_or_reported_decisions": int(multiplicity["tests"].sum()),
            "bh_survivors": int(multiplicity["bh_survivors"].sum()),
            "all_null_families": int(multiplicity["all_null"].sum()),
            "families_with_any_survivor": int((~multiplicity["all_null"]).sum()),
            "first_notebook": int(multiplicity["notebook_number"].min()),
            "last_notebook": int(multiplicity["notebook_number"].max()),
        }
    ]
)
display(family_summary)

,notebook_number,notebook,source_artifact,family,decision_column,tests,bh_survivors,all_null,source_sha256,manifest,manifest_sha256,seed,status
0,3,final_experiments/03_aggregation.ipynb,final_experiments/outputs/03_aggregation/devel...,nine aggregators × primary horizon h1; n-bin s...,bh_reject_q05,9,1,False,5ef91393609680a54bf59280e9c2dbb5e1d992f5ccbb4c...,final_experiments/outputs/03_aggregation/manif...,e23514f9ecaaffb953afa3c4eec3e486d501b7768e0adf...,20260731,exploratory_repaired_2026-07-31
1,4,final_experiments/04_surprise.ipynb,final_experiments/outputs/04_surprise/oos_r2_b...,four sentiment model additions vs control-only...,bh_reject_q05,5,2,False,d596bf97b2e12a06983a5ef6daa7492f7457e8dca3bc92...,final_experiments/outputs/04_surprise/manifest...,601838f09177032a7b8db421395a64c009fcc971e8a705...,none recorded / deterministic,exploratory_repaired_2026-07-31
2,7,final_experiments/07_strategy_analysis.ipynb,final_experiments/outputs/07_strategy_analysis...,4 displayed signals x 20 event-time lags,bh_reject_q05,80,0,True,5e3d17e72030ec50d3551afe62a4ed89f97f319951acb5...,final_experiments/outputs/07_strategy_analysis...,fac68d9cb8a1c37fb5aa4871675ee6ef107c5b4d5edfc1...,none recorded / deterministic,iterative_post_change_recompute_2026-08-03
3,8,final_experiments/08_story_type.ipynb,final_experiments/outputs/08_story_type/develo...,12 dominant event types x h1; BH-FDR q=0.05,bh_reject_q05,12,0,True,e297533ad54a45c9c8faeb99e00b7083cf53de62f10449...,final_experiments/outputs/08_story_type/manife...,1988e18aa5e3c6f75d2b8ffb89c3ff917d11d04bd43184...,20260731,PROVISIONAL_MACHINE_CLASSIFICATION
4,9,final_experiments/09_earnings.ipynb,final_experiments/outputs/09_earnings/developm...,pre/event/post interactions vs outside x h1; B...,bh_reject_q05,6,0,True,10219dd653a7b8b90cb33b5d55083ecf769a6a85946aeb...,final_experiments/outputs/09_earnings/manifest...,608eb134aa83387979b5a4fd1e421a24ee0f09d516ab7e...,none recorded / deterministic,aggregate result
5,11,final_experiments/11_lseg_robustness.ipynb,final_experiments/outputs/11_lseg_robustness/m...,two midcap-22 Reuters-only arms x h1; BH-FDR q...,bh_reject_q05,2,0,True,823858f2ae7d0741af19887ff9e396376639fa780fb1f9...,final_experiments/outputs/11_lseg_robustness/m...,cd857751677c9477702c66eb5c9d702adaf7a257ae4ce6...,none recorded / deterministic,NON_POOLED_ROBUSTNESS
6,11,final_experiments/11_lseg_robustness.ipynb,final_experiments/outputs/11_lseg_robustness/s...,five LSEG sector-33 source/actionability arms ...,bh_reject_q05,5,0,True,fcf4e8ec25deee250a65d466d925d35996d2fa3970892d...,final_experiments/outputs/11_lseg_robustness/m...,cd857751677c9477702c66eb5c9d702adaf7a257ae4ce6...,none recorded / deterministic,NON_POOLED_ROBUSTNESS
7,13,final_experiments/13_lseg_44_gemma_robustness....,final_experiments/outputs/13_lseg_44_gemma_rob...,two scorers x four pre-declared metadata filte...,bh_reject_q05,8,0,True,55e5c87fd1b0df02bcf90686e59966640b0b081858e471...,final_experiments/outputs/13_lseg_44_gemma_rob...,5c8fd8aec392da4a3ab416a980a67c0a6c33798a6c12bd...,20260805,RETROSPECTIVE_EXPLORATORY_NON_POOLED_ROBUSTNESS
8,13,final_experiments/13_lseg_44_gemma_robustness....,final_experiments/outputs/13_lseg_44_gemma_rob...,two scorers x nine pre-existing aggregators x ...,bh_reject_ic_q05,18,0,True,f3050a98d6c99d391f57a6b2d210dcf8fb268534c13efa...,final_experiments/outputs/13_lseg_44_gemma_rob...,5c8fd8aec392da4a3ab416a980a67c0a6c33798a6c12bd...,20260805,RETROSPECTIVE_EXPLORATORY_NON_POOLED_ROBUSTNESS
9,14,final_experiments/14_lseg_sparse_event_strateg...,final_experiments/outputs/14_lseg_sparse_event...,two predeclared sparse signals x three state h...,bh_reject_ic_q05,6,0,True,b344873f61ec4690e77880c3e249ab14707bb691e68f0f...,final_experiments/outputs/14_lseg_sparse_event...,85ccdbf88dcca60839eb026504e125c5fd8956a2a55467...,20260805,completed


,families,tests_or_reported_decisions,bh_survivors,all_null_families,families_with_any_survivor,first_notebook,last_notebook
0,56,264,27,36,20,3,73


## Core write-up number ledger

These are the principal numbers currently used to motivate the Gate-F1
framing and its limitations. Every row points at a saved artifact. The full
machine-readable family ledger is appended afterwards, so all reported tests
remain auditable even when they are not individually printed in the chapter.

In [3]:
core_specs = [
    {
        "claim": "FNSPID panel dimensions",
        "value": "715,546 firm-days; 570 symbols; 3,262 sessions",
        "artifact": "final_experiments/outputs/01_panel/manifest.json",
        "field": "n_rows / n_symbols / n_sessions",
    },
    {
        "claim": "FNSPID development negative-share IC",
        "value": "IC -0.005433; HAC t=-3.119",
        "artifact": "final_experiments/outputs/03_aggregation/development_ic_primary_ranked.csv",
        "field": "negative_share.ic",
    },
    {
        "claim": "FNSPID conditional negative-share coefficient",
        "value": "beta -0.00831 [-0.01410,-0.00252]; p=0.0049; 2,264 sessions",
        "artifact": "final_experiments/outputs/71_conditional_negative_share_transfer/conditional_coefficients.csv",
        "field": "FNSPID development estimate and HAC interval",
    },
    {
        "claim": "FNSPID reversal-controlled negative-share coefficient",
        "value": "beta -0.00914 [-0.01466,-0.00362]; p=0.00118; 2,264 sessions",
        "artifact": "final_experiments/outputs/79_fnspid_development_reversal_confound/primary_gate.csv",
        "field": "full price-path controls primary coefficient",
    },
    {
        "claim": "Conditional-effect yearly direction",
        "value": "negative in 8/9 development years",
        "artifact": "final_experiments/outputs/74_conditional_and_pressure_robustness_diagnostics/yearly_negative_share_coefficients.csv",
        "field": "sign of yearly estimate",
    },
    {
        "claim": "Conditional-effect multi-story diagnostic",
        "value": "n>=2 beta -0.00529 [-0.01272,+0.00214]",
        "artifact": "final_experiments/outputs/74_conditional_and_pressure_robustness_diagnostics/story_count_strata.csv",
        "field": "n>=2 firm-day stratum",
    },
    {
        "claim": "Development double-sort economic magnitude",
        "value": "all -0.62 [-1.51,+0.27]; n>=2 -0.25 [-1.82,+1.32] bps/session",
        "artifact": "final_experiments/outputs/76_fnspid_development_negative_share_double_sort/pooled_spread_summary.csv",
        "field": "all_firm_days mean and HAC interval",
    },
    {
        "claim": "Direct-signal turnover and break-even",
        "value": "about 200x/year; best break-even 0.625 bps/side",
        "artifact": "final_experiments/outputs/03_aggregation/development_ic_primary_ranked.csv",
        "field": "annualized_turnover / breakeven_bps_per_side",
    },
    {
        "claim": "Learned thresholds versus historical fixed band",
        "value": "-7.71 to -16.88 net bps/session; MDE 4.59 to 10.04 bps",
        "artifact": "final_experiments/outputs/77_null_minimum_detectable_effects/minimum_detectable_effects.csv",
        "field": "Learned trade/no-trade thresholds family",
    },
    {
        "claim": "HAR benchmark evaluation performance",
        "value": "Sharpe 0.673; return +29.85%; drawdown -15.73%",
        "artifact": "final_experiments/outputs/21_fnspid_sentiment_volatility_target/strategy_results.csv",
        "field": "control_har_target",
    },
    {
        "claim": "HAR x sentiment evaluation performance",
        "value": "Sharpe 0.835; return +35.20%; drawdown -16.36%",
        "artifact": "final_experiments/outputs/24_fnspid_har_sentiment_hysteresis/results.csv",
        "field": "har_sentiment",
    },
    {
        "claim": "HAR x sentiment downside inference",
        "value": "p=0.0148",
        "artifact": "final_experiments/outputs/24_fnspid_har_sentiment_hysteresis/paired_inference.csv",
        "field": "HAR minus HAR sentiment downside squared return",
    },
    {
        "claim": "Walk-forward HAR downside result",
        "value": "drawdown -12.53% to -8.57%; p=0.0052",
        "artifact": "final_experiments/outputs/25_fnspid_har_sentiment_walkforward/paired_inference.csv",
        "field": "HAR minus sentiment downside squared return",
    },
    {
        "claim": "FNSPID evaluation downside timing",
        "value": "q=0.0301",
        "artifact": "final_experiments/outputs/45_sentiment_downside_timing_placebo/timing_results.csv",
        "field": "FNSPID 2020-2023 downside_q_bh",
    },
    {
        "claim": "LSEG hybrid opened/backward Sharpe",
        "value": "2.271 / -0.354",
        "artifact": "final_experiments/outputs/67_lseg_gemma_finbert_joined_long_window/window_results.csv",
        "field": "opened_2025_2026 / backward_2024_2025 sharpe_net",
    },
    {
        "claim": "LSEG conditional-transfer coefficients",
        "value": "backward -0.0286; recent -0.0157; both intervals cross zero",
        "artifact": "final_experiments/outputs/71_conditional_negative_share_transfer/conditional_coefficients.csv",
        "field": "pinned-FinBERT backward and recent estimates",
    },
    {
        "claim": "Recent LSEG pressure non-activation",
        "value": "0/167 risk-off sessions; maximum z=1.349 vs 1.5 entry",
        "artifact": "final_experiments/outputs/72_lseg_negative_pressure_har_transfer/pressure_audit.csv",
        "field": "risk_off_sessions / max pressure z / entry threshold",
    },
    {
        "claim": "Backward LSEG pressure activation",
        "value": "53/456 sessions; 18 entries",
        "artifact": "final_experiments/outputs/73_lseg_negative_pressure_har_backward_transfer/pressure_audit.csv",
        "field": "risk_off_sessions / decision_sessions / risk_off_entries",
    },
    {
        "claim": "Backward LSEG pressure downside inference",
        "value": "BH q=0.0336",
        "artifact": "final_experiments/outputs/73_lseg_negative_pressure_har_backward_transfer/primary_inference.csv",
        "field": "baseline minus challenger downside squared return",
    },
    {
        "claim": "Backward LSEG pressure timing",
        "value": "BH q=0.0482",
        "artifact": "final_experiments/outputs/73_lseg_negative_pressure_har_backward_transfer/timing_tests.csv",
        "field": "HAR minus overlay downside squared return",
    },
    {
        "claim": "LSEG conditional MDE relative to FNSPID",
        "value": "7.6x backward; 13.6x recent; story 18.80 bps; publisher 0.049 IC; earnings 15.52 bps",
        "artifact": "final_experiments/outputs/77_null_minimum_detectable_effects/minimum_detectable_effects.csv",
        "field": "nominal_mde_multiple_of_reference",
    },
]

provenance_rows: list[dict[str, object]] = []
for entry in core_specs:
    artifact = ROOT / entry["artifact"]
    if not artifact.exists():
        raise FileNotFoundError(artifact)
    notebook_path, number = notebook_for_artifact(artifact)
    manifest_path = artifact.parent / "manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    frozen_spec = str(manifest.get("frozen_spec", manifest.get("declared_spec", "")))
    frozen_path = ROOT / frozen_spec if frozen_spec else None
    provenance_rows.append(
        {
            **entry,
            "notebook_number": number,
            "notebook": notebook_path,
            "artifact_sha256": sha256(artifact),
            "manifest": str(manifest_path.relative_to(ROOT)) if manifest_path.exists() else "",
            "manifest_sha256": sha256(manifest_path) if manifest_path.exists() else "",
            "frozen_spec": frozen_spec,
            "frozen_spec_sha256": sha256(frozen_path) if frozen_path and frozen_path.exists() else "",
            "seed": seeds_from_manifest(manifest) if manifest else "none recorded / deterministic",
            "status": str(manifest.get("status", manifest.get("analysis_status", "aggregate result"))),
        }
    )

provenance = pd.DataFrame(provenance_rows).sort_values("notebook_number").reset_index(drop=True)
display(provenance)

,claim,value,artifact,field,notebook_number,notebook,artifact_sha256,manifest,manifest_sha256,frozen_spec,frozen_spec_sha256,seed,status
0,FNSPID panel dimensions,"715,546 firm-days; 570 symbols; 3,262 sessions",final_experiments/outputs/01_panel/manifest.json,n_rows / n_symbols / n_sessions,1,final_experiments/01_panel.ipynb,dc5d7433e41c2dfc93c036820528ea1d582eb2280c3e84...,final_experiments/outputs/01_panel/manifest.json,dc5d7433e41c2dfc93c036820528ea1d582eb2280c3e84...,,,none recorded / deterministic,aggregate result
1,FNSPID development negative-share IC,IC -0.005433; HAC t=-3.119,final_experiments/outputs/03_aggregation/devel...,negative_share.ic,3,final_experiments/03_aggregation.ipynb,07151a502bce0e6a8e37ce1c84103a6e52b356746b5766...,final_experiments/outputs/03_aggregation/manif...,e23514f9ecaaffb953afa3c4eec3e486d501b7768e0adf...,,,20260731,exploratory_repaired_2026-07-31
2,Direct-signal turnover and break-even,about 200x/year; best break-even 0.625 bps/side,final_experiments/outputs/03_aggregation/devel...,annualized_turnover / breakeven_bps_per_side,3,final_experiments/03_aggregation.ipynb,07151a502bce0e6a8e37ce1c84103a6e52b356746b5766...,final_experiments/outputs/03_aggregation/manif...,e23514f9ecaaffb953afa3c4eec3e486d501b7768e0adf...,,,20260731,exploratory_repaired_2026-07-31
3,HAR benchmark evaluation performance,Sharpe 0.673; return +29.85%; drawdown -15.73%,final_experiments/outputs/21_fnspid_sentiment_...,control_har_target,21,final_experiments/21_fnspid_sentiment_volatili...,471b008ef45bc2362b3d8989baf794a5ab76fa05c5c133...,final_experiments/outputs/21_fnspid_sentiment_...,33bd7ec5a7e9caac6dff56a72b15459889853ca897ff6e...,,,20260805,exploratory_frozen_before_notebook21_evaluatio...
4,HAR x sentiment downside inference,p=0.0148,final_experiments/outputs/24_fnspid_har_sentim...,HAR minus HAR sentiment downside squared return,24,final_experiments/24_fnspid_har_sentiment_hyst...,cf3a6fb033ddd22ac2b1471d007d6ebedb998ed0355357...,final_experiments/outputs/24_fnspid_har_sentim...,ea4bb749e01508d6024b1310390041a73f899fcc98e585...,,,20260805,iterative_evaluation_spec_frozen_before_notebo...
5,HAR x sentiment evaluation performance,Sharpe 0.835; return +35.20%; drawdown -16.36%,final_experiments/outputs/24_fnspid_har_sentim...,har_sentiment,24,final_experiments/24_fnspid_har_sentiment_hyst...,e33b8c4e01080d1d023efda7dd1e024b5df19610cf3b86...,final_experiments/outputs/24_fnspid_har_sentim...,ea4bb749e01508d6024b1310390041a73f899fcc98e585...,,,20260805,iterative_evaluation_spec_frozen_before_notebo...
6,Walk-forward HAR downside result,drawdown -12.53% to -8.57%; p=0.0052,final_experiments/outputs/25_fnspid_har_sentim...,HAR minus sentiment downside squared return,25,final_experiments/25_fnspid_har_sentiment_walk...,c9bbb3a2a24c4ccb5e2f3499d082087b02d1e492743213...,final_experiments/outputs/25_fnspid_har_sentim...,eb96a1b5069f9184e97b6a6e2724f6adedcd4f7b440aac...,,,20260805,retrospective_historical_robustness_frozen_bef...
7,FNSPID evaluation downside timing,q=0.0301,final_experiments/outputs/45_sentiment_downsid...,FNSPID 2020-2023 downside_q_bh,45,final_experiments/45_sentiment_downside_timing...,a0defab69499323c4949463c0516669cc91856905d0c35...,final_experiments/outputs/45_sentiment_downsid...,303b6bc40cc2e884ab1bef082803eb86ab3a2f882571f9...,,,20260805,complete_post_result_mechanism_audit
8,LSEG hybrid opened/backward Sharpe,2.271 / -0.354,final_experiments/outputs/67_lseg_gemma_finber...,opened_2025_2026 / backward_2024_2025 sharpe_net,67,final_experiments/67_lseg_gemma_finbert_joined...,ef80325cc8cc4b12c9956619ba010dfe8c17ce89f69773...,final_experiments/outputs/67_lseg_gemma_finber...,bee73f9e6f0a0de290dca3197590080480520e03d74cb8...,final_experiments/frozen_specs/lseg_gemma_finb...,5848dbac02b6eb33afe56c2df3680451aaecdc19f97f55...,20260820,POST_DRIFT_STOP_JOINED_LONG_WINDOW_MEASUREMENT...
9,FNSPID conditional negative-share coefficient,"beta -0.00831 [-0.01410,-0.00252]; p=0.0049; 2...",final_experiments/outputs/

## Persist CSV, Markdown and dissertation-ready LaTeX tables

In [4]:
multiplicity.to_csv(OUTPUT / "multiplicity_family_results.csv", index=False)
family_summary.to_csv(OUTPUT / "multiplicity_summary.csv", index=False)
provenance.to_csv(OUTPUT / "core_result_provenance.csv", index=False)

with (OUTPUT / "multiplicity_family_results.md").open("w") as handle:
    handle.write(markdown_table(multiplicity[["notebook_number", "family", "tests", "bh_survivors", "all_null"]]))
    handle.write("\n")
with (OUTPUT / "core_result_provenance.md").open("w") as handle:
    handle.write(markdown_table(provenance[["claim", "value", "notebook_number", "artifact", "artifact_sha256", "seed", "status"]]))
    handle.write("\n")

family_tex_rows = []
for row in multiplicity.itertuples():
    family_tex_rows.append(
        f"{row.notebook_number:02d} & {tex_escape(row.family)} & {row.tests:d} & {row.bh_survivors:d} \\\\"
    )
MULTIPLICITY_TEX.write_text(
    "\n".join(
        [
            "% Generated by Notebook 78; see its CSV and manifest for full paths and hashes.",
            r"\begingroup",
            r"\small",
            r"\begin{longtable}{r p{9.2cm} rr}",
            r"\caption{Pre-declared multiplicity families and BH/FDR outcomes}",
            r"\label{tab:multiplicity-families} \\",
            r"\toprule",
            r"Notebook & Family & Tests & Survivors \\",
            r"\midrule",
            r"\endfirsthead",
            r"\toprule",
            r"Notebook & Family & Tests & Survivors \\",
            r"\midrule",
            r"\endhead",
            *family_tex_rows,
            r"\bottomrule",
            r"\end{longtable}",
            r"\endgroup",
            "",
        ]
    )
)

provenance_tex_rows = []
for row in provenance.itertuples():
    short_hash = str(row.artifact_sha256)[:12]
    frozen_spec = "none" if pd.isna(row.frozen_spec) or not str(row.frozen_spec) else Path(str(row.frozen_spec)).stem
    frozen_spec_tex = tex_escape(frozen_spec).replace(r"\_", r"\_\allowbreak{}")
    seed = str(row.seed).replace("none recorded / deterministic", "none / deterministic")
    provenance_tex_rows.append(
        f"{tex_escape(row.claim)} & {row.notebook_number:02d} & {tex_escape(row.value)} & "
        f"{frozen_spec_tex} & {short_hash} & {tex_escape(seed)} \\\\"
    )
PROVENANCE_TEX.write_text(
    "\n".join(
        [
            "% Generated by Notebook 78; the full CSV carries paths, full hashes, specs, seeds and status.",
            r"\begingroup",
            r"\footnotesize",
            r"\begin{longtable}{p{3.1cm} r p{3.2cm} p{3.3cm} l p{1.8cm}}",
            r"\caption{Provenance of the core reported results}",
            r"\label{tab:core-result-provenance} \\",
            r"\toprule",
            r"Claim & NB & Reported value & Frozen spec & Artifact hash & Seed \\",
            r"\midrule",
            r"\endfirsthead",
            r"\toprule",
            r"Claim & NB & Reported value & Frozen spec & Artifact hash & Seed \\",
            r"\midrule",
            r"\endhead",
            *provenance_tex_rows,
            r"\bottomrule",
            r"\end{longtable}",
            r"\endgroup",
            "",
        ]
    )
)

git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=ROOT, check=True, capture_output=True, text=True
).stdout.strip()
manifest = {
    "notebook": "78_null_results_and_provenance_tables",
    "status": "complete_aggregate_only_examiner_audit",
    "git_commit_at_execution": git_commit,
    "declared_spec": str(SPEC_PATH.relative_to(ROOT)),
    "declared_spec_sha256": sha256(SPEC_PATH),
    "multiplicity_scope": spec["notebook_78"]["multiplicity_scope"],
    "multiplicity_summary": json.loads(family_summary.to_json(orient="records"))[0],
    "multiplicity_source_hashes": {
        path: digest
        for path, digest in multiplicity[["source_artifact", "source_sha256"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    },
    "core_provenance_rows": len(provenance),
    "core_artifact_hashes": dict(zip(provenance["artifact"], provenance["artifact_sha256"], strict=True)),
    "claim_boundary": spec["claim_boundary"],
}
(OUTPUT / "manifest.json").write_text(json.dumps(manifest, indent=2, allow_nan=True) + "\n")

summary = family_summary.iloc[0]
display(
    Markdown(
        f"""### Audit result

The machine-readable ledger contains **{int(summary['families'])} distinct
BH/FDR family records** and **{int(summary['tests_or_reported_decisions'])}
saved decisions** through Notebook 74. **{int(summary['bh_survivors'])}** survive;
**{int(summary['all_null_families'])}** families have no survivor.

The core provenance table maps **{len(provenance)}** current write-up claims to
full artifact hashes, manifests, frozen specs, seeds and status boundaries. The
unexecuted Notebook 75 is absent by construction.
"""
    )
)

### Audit result

The machine-readable ledger contains **56 distinct
BH/FDR family records** and **264
saved decisions** through Notebook 74. **27** survive;
**36** families have no survivor.

The core provenance table maps **21** current write-up claims to
full artifact hashes, manifests, frozen specs, seeds and status boundaries. The
unexecuted Notebook 75 is absent by construction.
